# Collect agent traces on a Colab T4 (Ollama)

Launcher only: every step calls a CLI from the `atl` package. No logic lives here.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU.*
2. Add the Colab secret `GEMINI_API_KEY` (key icon in the left bar) and allow this notebook to use it.
   The local model is the agent; the user simulator still runs on Gemini (small cost, logged).

Outputs, the LLM cache and the cost log live on Google Drive, so a dead session loses at most the
run in progress: re-run all cells and collection resumes.

In [ ]:
# Pick the run config (one local model per run).
CONFIG = "configs/agent/bench_retail_qwen3_8b.yaml"  # or bench_retail_llama31_8b.yaml
DRIVE_DATA = "/content/drive/MyDrive/agent-trust-layer/data"
REPO = "https://github.com/Vivekananda-S/agent-trust-layer.git"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "$DRIVE_DATA"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d agent-trust-layer && git -C agent-trust-layer pull --ff-only || git clone -q "$REPO"
%cd /content/agent-trust-layer
!python --version
!pip install -q -e ".[agent]"
# data/ is gitignored: point it at Drive so traces, cache and cost log survive the session.
!test -L data || (rm -rf data && ln -s "$DRIVE_DATA" data) && ls -la data

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
# Install and start Ollama with a 32k default context (the run config also sets num_ctx).
!apt-get -qq install -y zstd pciutils > /dev/null
!command -v ollama || (curl -fsSL https://ollama.com/install.sh | sh)
import subprocess, time, urllib.request
env = {**os.environ, "OLLAMA_CONTEXT_LENGTH": "32768", "OLLAMA_KEEP_ALIVE": "-1"}
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("ollama up"); break
    except Exception:
        time.sleep(1)

In [ ]:
# Pull the model named in the config (e.g. ollama_chat/qwen3:8b -> qwen3:8b).
import yaml
MODEL_TAG = yaml.safe_load(open(CONFIG))["agent_model"].split("/", 1)[1]
!ollama pull "$MODEL_TAG"
!ollama list

In [ ]:
# tau2 data at the pinned commit, then the offline smoke run ($0, fake LLM).
!atl-agent fetch-data
!atl-agent run --config "$CONFIG" --smoke --smoke-dir /content/smoke

In [ ]:
# The real run. Resumes automatically; stops cleanly on budget, context overflow or quota errors.
!mkdir -p data/logs
!atl-agent run --config "$CONFIG" 2>&1 | tee -a data/logs/$(basename "$CONFIG" .yaml).log | grep -E "atl\.|Traceback|Error" 

In [ ]:
# Throughput, success rate and cost for this run; check the GPU was actually used.
RUN_NAME = yaml.safe_load(open(CONFIG))["run_name"]
!atl-agent stats --run-dir "data/traces/$RUN_NAME"
!ollama ps
!tail -n 5 /content/ollama.log